In [1]:
from structure_tensor import eig_special_3d, structure_tensor_3d
import numpy as np

In [2]:
np.random.seed(0)

shape = (64, 64, 64)
volume = np.random.randn(*shape).astype(np.float32)

In [3]:
S = structure_tensor_3d(volume, sigma=1.0, rho=2.0)

Sxx, Syy, Szz, Sxy, Sxz, Syz = S

In [4]:
print(S[:,0,0,0])

[0.06361035 0.06808753 0.07159885 0.05347546 0.03794962 0.03946569]


In [13]:
# ------------------------------------------------------------
# 3. evaluate isotropy
# ------------------------------------------------------------
def stats(name, arr):
    return f"{name}: mean={arr.mean():.4e}, std={arr.std():.4e}"

print("=== Diagonal ===")
print(stats("Sxx", Sxx))
print(stats("Syy", Syy))
print(stats("Szz", Szz))

print("\n=== Off-diagonal ===")
print(stats("Sxy", Sxy))
print(stats("Sxz", Sxz))
print(stats("Syz", Syz))

# ------------------------------------------------------------
# 4. check isotropy numerically
# ------------------------------------------------------------
diag_mean = np.mean([Sxx.mean(), Syy.mean(), Szz.mean()])
off_mean = np.mean([np.abs(Sxy).mean(), np.abs(Sxz).mean(), np.abs(Syz).mean()])

print("\n=== Isotropy metrics ===")
print(f"diag similarity: {np.std([Sxx.mean(), Syy.mean(), Szz.mean()]):.4e}")
print(f"off-diagonal magnitude: {off_mean:.4e}")

=== Diagonal ===
Sxx: mean=1.2415e-02, std=3.9734e-03
Syy: mean=1.2638e-02, std=4.1619e-03
Szz: mean=1.2484e-02, std=4.2928e-03

=== Off-diagonal ===
Sxy: mean=-3.3842e-06, std=2.0776e-03
Sxz: mean=5.5570e-06, std=2.0143e-03
Syz: mean=-9.4695e-05, std=2.1681e-03

=== Isotropy metrics ===
diag similarity: 9.3183e-05
off-diagonal magnitude: 1.4492e-03


In [ ]:

# ------------------------------------------------------------
# # helper: convert full symmetric 3x3 matrix -> packed form
# # expected by eig_special_3d
# # ------------------------------------------------------------
# def pack_sym33(A):
#     return np.array([
#         A[0, 0],  # Sxx
#         A[1, 1],  # Syy
#         A[2, 2],  # Szz
#         A[0, 1],  # Sxy
#         A[0, 2],  # Sxz
#         A[1, 2],  # Syz
#     ], dtype=np.float32).reshape(6, 1)


# # ------------------------------------------------------------
# # build a nearly degenerate tensor
# # ------------------------------------------------------------
# eps = 1.
# A = np.array([
#     [1.0,   eps,   eps],
#     [eps,   1.0,   eps],
#     [eps,   eps,   1.0]
# ], dtype=np.float64)

# S = pack_sym33(A)

# ------------------------------------------------------------
# call eig_special_3d
# ------------------------------------------------------------
val_st, vec_st = eig_special_3d(S, full=False, eigenvalue_order="asc")

print("=== eig_special_3d ===")
print("eigenvalues:", val_st[:, 0])
print("eigenvector:", vec_st[:, 0])
print("norm(vec):", np.linalg.norm(vec_st[:, 0]))

# ------------------------------------------------------------
# compare to numpy.linalg.eigh
# ------------------------------------------------------------
w_np, v_np = np.linalg.eigh(A)

print("\n=== numpy.linalg.eigh ===")
print("eigenvalues:", w_np)
print("eigenvector for smallest eigenvalue:", v_np[:, 0])
print("norm(vec):", np.linalg.norm(v_np[:, 0]))

# ------------------------------------------------------------
# inspect the raw analytic vector construction before normalization
# copied from eig_special_3d for the single-vector case
# ------------------------------------------------------------
l = val_st[0, 0]   # smallest eigenvalue for asc

u = A[0, 2] * A[1, 2] - (A[2, 2] - l) * A[0, 1]
v = A[0, 1] * A[1, 2] - (A[1, 1] - l) * A[0, 2]
w = A[0, 1] * A[0, 2] - (A[0, 0] - l) * A[1, 2]

raw_vec = np.array([
    u * v,
    u * w,
    v * w
], dtype=np.float32)

print("\n=== raw vector before normalization (analytic construction) ===")
print("raw_vec:", raw_vec)
print("raw norm:", np.linalg.norm(raw_vec))

=== eig_special_3d ===
eigenvalues: [[[0.01232647 0.01611362 0.0174255  ... 0.00822095 0.00840757 0.00846761]
  [0.01058664 0.01404829 0.01616371 ... 0.00825718 0.0079027  0.00767042]
  [0.00904948 0.01190556 0.01449888 ... 0.00821485 0.00741027 0.00703649]
  ...
  [0.01674854 0.01816835 0.01847054 ... 0.01055275 0.01069823 0.01079757]
  [0.0165405  0.01979438 0.02160092 ... 0.01135316 0.01160015 0.01139478]
  [0.01852992 0.02337496 0.02615539 ... 0.01163419 0.01222219 0.01172185]]

 [[0.03575029 0.03337175 0.02866077 ... 0.01495567 0.01728813 0.01910711]
  [0.04040868 0.03639495 0.02996641 ... 0.0153933  0.01916523 0.02263782]
  [0.04802998 0.04194058 0.03305362 ... 0.01469775 0.01891328 0.02295754]
  ...
  [0.02622086 0.02668475 0.0263875  ... 0.0235448  0.03524533 0.04787557]
  [0.02683493 0.03012204 0.03150874 ... 0.02288442 0.03419128 0.04651166]
  [0.02740019 0.03316037 0.03600452 ... 0.01905306 0.02755241 0.03721723]]

 [[0.15521996 0.13749531 0.12287344 ... 0.05848662 0.0880458